In [1]:
%run ./03_1_dq_rule_engine

StatementMeta(, 454d0e43-94d3-445e-89a9-3e39138ad147, 8, Finished, Available, Finished, True)

In [2]:
%run ./03_1_1_dq_rule_config

StatementMeta(, 454d0e43-94d3-445e-89a9-3e39138ad147, 9, Finished, Available, Finished, True)

[DQ CONFIG] loaded 13 rules across 5 tables.


In [3]:
from notebookutils import mssparkutils
from pyspark.sql import functions as F
from datetime import datetime

try:
    batch_id = mssparkutils.widgets.get("batch_id")
except Exception:
    batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

DB = "yelp_lakehouse.dbo"
LAYER = "silver"
DEBUG = True

dq_rule_tbl = f"{DB}.dq_rule_result"
dq_gate_tbl = f"{DB}.dq_table_gate"

pipeline_run_id = batch_id
run_ts = datetime.now()

print(f"batch_id = {batch_id}")

StatementMeta(, 454d0e43-94d3-445e-89a9-3e39138ad147, 10, Finished, Available, Finished, False)

batch_id = 20260417_194857


In [4]:
all_rule_results = []
all_gate_results = []

for table_name, rules in DQ_CONFIG.items():
    df = spark.table(f"{DB}.{table_name}")
    dq_run_id = new_dq_run_id()

    results = []

    for rule in rules:
        if rule["rule_type"] == "duplicate_key":
            r = eval_duplicate_key_rule(
            df = df,
            rule = rule,
            dq_run_id = dq_run_id,
            pipeline_run_id = pipeline_run_id,
            layer = LAYER,
            table_name = table_name,
            run_ts = run_ts
            )
        else:
            r = eval_standard_rule(
                df = df,
                rule = rule,
                dq_run_id = dq_run_id,
                pipeline_run_id = pipeline_run_id,
                layer = LAYER,
                table_name = table_name,
                run_ts = run_ts
            )
        results.append(r)
    
    gate = build_gate_result(
        results,
        dq_run_id,
        pipeline_run_id,
        LAYER,
        table_name,
        run_ts
    )

    all_rule_results.extend(results)
    all_gate_results.append(gate)

StatementMeta(, 454d0e43-94d3-445e-89a9-3e39138ad147, 11, Finished, Available, Finished, False)

In [5]:
rule_df = spark.createDataFrame(all_rule_results)
gate_df = spark.createDataFrame(all_gate_results)
(
    rule_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(dq_rule_tbl)
)

(
    gate_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(dq_gate_tbl)
)

StatementMeta(, 454d0e43-94d3-445e-89a9-3e39138ad147, 12, Finished, Available, Finished, False)

In [6]:
if DEBUG:
    display(rule_df)
    display(gate_df)

StatementMeta(, 454d0e43-94d3-445e-89a9-3e39138ad147, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b3da92dc-551d-45a5-bd9d-6f8c42f69549)

SynapseWidget(Synapse.DataFrame, 30127f9e-9ee8-4c1e-9e07-a462de2fcce5)

In [7]:
# Gate decision: determine overall status across all tables
gate_decisions = [g["decision"] for g in all_gate_results]

if "BLOCKED" in gate_decisions:
    overall_gate = "BLOCKED"
elif "DEGRADED" in gate_decisions:
    overall_gate = "DEGRADED"
else:
    overall_gate = "PASS"

print(f"[DQ] Overall gate decision: {overall_gate}")
mssparkutils.notebook.exit(overall_gate)

StatementMeta(, 454d0e43-94d3-445e-89a9-3e39138ad147, 14, Finished, Available, Finished, False)

[DQ] Overall gate decision: PASS
ExitValue: PASS